[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_06/notebook_6_2_melanoma_classification_fairness.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 6.2: Melanoma Classification with Fairness Analysis

**Chapter 6: Medical Imaging - Implementing Journey 5 (Priya)**

**Journey Connection**: This notebook implements Journey 5 from Chapter 3, where Priya's melanoma was missed by an AI system that performed poorly on darker skin tones. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Generate synthetic skin lesion images with varying skin tones
2. Apply transfer learning (pretrain a CNN, freeze its convolutional layers, fine-tune a new head) and compare it with a classical baseline
3. Identify performance disparities across demographic groups
4. Calculate fairness metrics (equalized odds, demographic parity)
5. Understand how training data bias leads to clinical harm

## Clinical Context

Melanoma is the deadliest form of skin cancer. Early detection dramatically improves outcomes. AI systems show promise for screening, but performance varies by skin tone.

**Priya's story** (Chapter 3, Journey 5): Priya, 34, has medium-dark skin (Fitzpatrick type IV). She noticed a changing mole on her shoulder and tried a smartphone app that claimed "dermatologist-level accuracy". App result: *"Low risk, benign nevus. Confidence: 87%."* Reassured, she skipped the dermatologist. Three months later the mole had grown and developed an irregular border; biopsy showed **melanoma, stage IB (1.2 mm depth)**, needing wider surgical margins and with a worse prognosis than if caught three months earlier.

**The problem** (figures from the book):
- Training data: 92% Fitzpatrick I-III, only 8% Fitzpatrick IV-VI
- Marketed "95% sensitivity" overall, but by skin type: **94%** (I-II), **82%** (III-IV), **67%** (V-VI)
- This is algorithmic bias leading to health disparities

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Generate Synthetic Skin Lesion Data

We'll create simplified skin lesion images with:
- Different Fitzpatrick skin types (I-VI)
- Benign lesions (nevi, seborrheic keratosis)
- Malignant lesions (melanoma)

Key challenges:
- **Lower lesion contrast on darker skin**: a pigmented lesion is darker than the skin around it, so on darker skin the lesion and its colour variation stand out less against the same camera noise
- **Training data imbalance** by skin tone (Section 2)

All images are synthetic, with variable lesion size, shape, pigment and variegation, so the task is deliberately not trivial.

In reality, you would use datasets like ISIC, HAM10000, or Fitzpatrick17k.

In [3]:
from scipy.ndimage import gaussian_filter

# Skin tone colours (RGB) by Fitzpatrick type
SKIN_TONES = {
    1: (252, 233, 219),  # Very light
    2: (242, 213, 189),  # Light
    3: (224, 186, 154),  # Light-medium
    4: (193, 145, 105),  # Medium
    5: (141,  99,  63),  # Medium-dark
    6: ( 97,  62,  32),  # Dark
}

def generate_synthetic_lesion(size=64, skin_type=3, is_malignant=False):
    """
    Generate a synthetic pigmented skin lesion (RGB, values in [0, 1]).

    Mechanism behind the skin-tone effect (nothing is hard-coded per group):
    - A pigmented lesion is modelled as the surrounding skin DARKENED by a factor,
      so its absolute contrast against the skin shrinks on darker skin.
    - Camera auto-exposure brightens darker scenes, but only partly (exponent 0.6),
      so darker skin still ends up with less lesion contrast in the photo.
    - Sensor/compression noise has the same absolute size for every skin type, so on
      darker skin the melanoma's colour variegation is closer to the noise level.

    Parameters:
    - size: Image dimensions (size x size x 3)
    - skin_type: Fitzpatrick type (1-6)
    - is_malignant: Whether lesion is melanoma
    """
    base = np.array(SKIN_TONES[skin_type], dtype=float) / 255.0
    # Partial auto-exposure: gain (0.6 / mean brightness)^0.6 (1 = full correction, 0 = none)
    base = np.clip(base * (0.6 / base.mean()) ** 0.6, 0, 1)
    yy, xx = np.mgrid[:size, :size]

    # Skin: mild illumination gradient + sensor noise
    gradient = 1 + 0.08 * ((xx / size - 0.5) * np.random.uniform(-1, 1) +
                           (yy / size - 0.5) * np.random.uniform(-1, 1))
    image = base[None, None, :] * gradient[..., None] + np.random.normal(0, 0.05, (size, size, 3))

    # Lesion shape: ellipse of random size, position and elongation
    r = np.random.uniform(size * 0.12, size * 0.25)
    cy, cx = size / 2 + np.random.uniform(-6, 6, 2)
    ecc = np.random.uniform(0.7, 1.3)
    ellipse = ((yy - cy) / (r * ecc))**2 + ((xx - cx) / (r / ecc))**2
    lesion_mask = ellipse <= 1

    darken = np.random.uniform(0.45, 0.8)              # amount of pigment
    if is_malignant:
        # B: Border irregularity (in 60% of melanomas)
        if np.random.rand() < 0.6:
            lesion_mask = ellipse <= 1 + 1.5 * gaussian_filter(np.random.randn(size, size), 3)
        # C: Colour variegation, from subtle to marked
        variegation = np.random.uniform(0.18, 0.4)
    else:
        # Benign nevi: mostly uniform, some slightly variegated
        variegation = np.random.uniform(0.0, 0.08)

    shade = np.clip(darken + 3 * variegation * gaussian_filter(np.random.randn(size, size), 1.5), 0.2, 1.0)
    for c in range(3):
        image[..., c][lesion_mask] = (base[c] * shade)[lesion_mask] + np.random.normal(0, 0.05, lesion_mask.sum())

    return np.clip(image, 0, 1)

# Generate examples
print("Generating example skin lesion images...")

examples = []
for skin_type in [1, 3, 6]:
    for is_malignant in [False, True]:
        img = generate_synthetic_lesion(size=64, skin_type=skin_type, is_malignant=is_malignant)
        examples.append((img, skin_type, is_malignant))

print(f"✓ Generated {len(examples)} example images")

Generating example skin lesion images...
✓ Generated 6 example images


In [4]:
# Visualize examples
fig, axes = plt.subplots(3, 2, figsize=(8, 12))

skin_type_names = {1: 'Type I (Very Light)', 3: 'Type III (Medium)', 6: 'Type VI (Dark)'}

idx = 0
for i, skin_type in enumerate([1, 3, 6]):
    # Benign
    axes[i, 0].imshow(examples[idx][0])
    axes[i, 0].set_title(f'{skin_type_names[skin_type]}\nBenign Nevus', fontweight='bold')
    axes[i, 0].axis('off')
    idx += 1

    # Malignant
    axes[i, 1].imshow(examples[idx][0])
    axes[i, 1].set_title(f'{skin_type_names[skin_type]}\nMelanoma', fontweight='bold', color='red')
    axes[i, 1].axis('off')
    idx += 1

plt.tight_layout()
plt.show()

print("\n📊 Observation:")
print("   - Melanoma shows asymmetry, irregular borders, color variation")
print("   - These features are HARDER to detect on darker skin")
print("   - Color contrast is reduced on darker backgrounds")


📊 Observation:
   - Melanoma shows asymmetry, irregular borders, color variation
   - These features are HARDER to detect on darker skin
   - Color contrast is reduced on darker backgrounds


## 2. Generate Biased Training Dataset

This is the critical problem: training data reflects historical biases.

**Distribution used here** (matching the app in Priya's story: 92% types I-III, 8% types IV-VI):

| Fitzpatrick type | I | II | III | IV | V | VI |
|---|---|---|---|---|---|---|
| Share of training images | 35% | 35% | 22% | 4% | 2% | 2% |

So 70% light (I-II), 26% medium (III-IV) and only 4% dark (V-VI). This mirrors the problem in datasets like ISIC.

In [5]:
def generate_biased_dataset(n_samples=1000, melanoma_rate=0.20):
    """
    Generate dataset with realistic skin type distribution.

    Mirrors real-world bias in dermatology datasets.
    """

    dataset = []

    # Biased skin type distribution
    # 92% types I-III, 8% types IV-VI (as in Priya's app, Chapter 3)
    skin_type_probs = {
        1: 0.35,  # Very light
        2: 0.35,  # Light
        3: 0.22,  # Medium-light
        4: 0.04,  # Medium
        5: 0.02,  # Medium-dark
        6: 0.02,  # Dark
    }

    for i in range(n_samples):
        # Sample skin type according to biased distribution
        skin_type = np.random.choice(list(skin_type_probs.keys()),
                                    p=list(skin_type_probs.values()))

        # Determine if malignant
        is_malignant = np.random.rand() < melanoma_rate

        # Generate image
        img = generate_synthetic_lesion(size=64, skin_type=skin_type, is_malignant=is_malignant)

        # Extract features (simple color histograms for this educational example)
        # In reality, use CNN features
        features = []
        for c in range(3):  # RGB channels
            hist, _ = np.histogram(img[:, :, c], bins=8, range=(0, 1))
            features.extend(hist / hist.sum())

        # Add texture features
        features.append(np.std(img))  # Overall variation
        features.append(np.std(img[:, :, 0] - img[:, :, 1]))  # Color contrast

        dataset.append({
            'features': np.array(features),
            'label': 1 if is_malignant else 0,
            'skin_type': skin_type,
            'skin_group': 'Light (I-II)' if skin_type <= 2 else
                         'Medium (III-IV)' if skin_type <= 4 else
                         'Dark (V-VI)',
            'image': img
        })

    return pd.DataFrame(dataset)

# Generate dataset
print("Generating biased training dataset...")
df_train = generate_biased_dataset(n_samples=800, melanoma_rate=0.20)

# Generate balanced test set (to reveal bias)
print("Generating balanced test dataset...")
df_test = generate_biased_dataset(n_samples=300, melanoma_rate=0.20)

# Force balanced skin type distribution in test set
test_samples_per_type = 50
df_test_balanced = []
for skin_type in range(1, 7):
    for is_malignant in [0, 1]:
        for _ in range(test_samples_per_type // 2):
            img = generate_synthetic_lesion(size=64, skin_type=skin_type, is_malignant=bool(is_malignant))
            features = []
            for c in range(3):
                hist, _ = np.histogram(img[:, :, c], bins=8, range=(0, 1))
                features.extend(hist / hist.sum())
            features.append(np.std(img))
            features.append(np.std(img[:, :, 0] - img[:, :, 1]))

            df_test_balanced.append({
                'features': np.array(features),
                'label': is_malignant,
                'skin_type': skin_type,
                'skin_group': 'Light (I-II)' if skin_type <= 2 else
                             'Medium (III-IV)' if skin_type <= 4 else
                             'Dark (V-VI)',
                'image': img
            })

df_test = pd.DataFrame(df_test_balanced)

print(f"\nTraining set: {len(df_train)} samples")
print(f"  Melanoma rate: {df_train['label'].mean()*100:.1f}%")
print(f"\nSkin type distribution (training):")
print(df_train['skin_group'].value_counts())
print(f"\nTest set: {len(df_test)} samples (balanced by skin type)")
print(df_test['skin_group'].value_counts())

Generating biased training dataset...


Generating balanced test dataset...



Training set: 800 samples
  Melanoma rate: 23.1%

Skin type distribution (training):
skin_group
Light (I-II)       575
Medium (III-IV)    194
Dark (V-VI)         31
Name: count, dtype: int64

Test set: 300 samples (balanced by skin type)
skin_group
Light (I-II)       100
Medium (III-IV)    100
Dark (V-VI)        100
Name: count, dtype: int64


In [6]:
# Visualize dataset bias
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set distribution
ax = axes[0]
train_dist = df_train['skin_group'].value_counts()
ax.bar(range(len(train_dist)), train_dist.values, color=['#FFE4C4', '#DEB887', '#8B4513'])
ax.set_xticks(range(len(train_dist)))
ax.set_xticklabels(train_dist.index, rotation=15, ha='right')
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Training Set: Biased Distribution\n(Mirrors Real Datasets)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add percentages
for i, v in enumerate(train_dist.values):
    ax.text(i, v + 10, f'{v/len(df_train)*100:.0f}%', ha='center', fontweight='bold')

# Test set distribution
ax = axes[1]
test_dist = df_test['skin_group'].value_counts()
ax.bar(range(len(test_dist)), test_dist.values, color=['#FFE4C4', '#DEB887', '#8B4513'])
ax.set_xticks(range(len(test_dist)))
ax.set_xticklabels(test_dist.index, rotation=15, ha='right')
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Test Set: Balanced Distribution\n(To Reveal Bias)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n⚠️  CRITICAL OBSERVATION:")
train_share = df_train['skin_group'].value_counts(normalize=True)
print(f"   Training data is {train_share.get('Light (I-II)', 0):.0%} light skin, "
      f"only {train_share.get('Dark (V-VI)', 0):.0%} dark skin")
print(f"   Types I-III: {(df_train['skin_type'] <= 3).mean():.0%}, types IV-VI: {(df_train['skin_type'] >= 4).mean():.0%} "
      "(book: 92% / 8%)")
print("   Predict: will the model perform worse on dark skin? We check this below.")
print("   Under-representation like this is the problem that affected Priya.")


⚠️  CRITICAL OBSERVATION:
   Training data is 72% light skin, only 4% dark skin
   Types I-III: 93%, types IV-VI: 7% (book: 92% / 8%)
   Predict: will the model perform worse on dark skin? We check this below.
   Under-representation like this is the problem that affected Priya.


## 3. Train Classifier

We start with a simple baseline: a RandomForest on hand-crafted colour/texture features.
`class_weight='balanced'` compensates for melanomas being only ~20% of the images (in screening, missed cancers matter most).
In Section 7 we train a CNN with **transfer learning** and compare it with this baseline.

In [7]:
# Prepare data
X_train = np.stack(df_train['features'].values)
y_train = df_train['label'].values

X_test = np.stack(df_test['features'].values)
y_test = df_test['label'].values

print(f"Feature matrix shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

# Train classifier
print(f"\nTraining Random Forest classifier...")
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

# Predictions
y_train_pred = clf.predict(X_train)
y_train_proba = clf.predict_proba(X_train)[:, 1]

y_test_pred = clf.predict(X_test)
y_test_proba = clf.predict_proba(X_test)[:, 1]

# Overall performance
print(f"\n" + "="*70)
print(f"OVERALL PERFORMANCE")
print("="*70)
print(f"Training Accuracy: {accuracy_score(y_train, y_train_pred):.3f}")
print(f"Test Accuracy:     {accuracy_score(y_test, y_test_pred):.3f}")
print(f"Test AUROC:        {roc_auc_score(y_test, y_test_proba):.3f}")
print("="*70)

print("\n✓ Model trained successfully")
print("Now let's examine performance BY SKIN TYPE...")

Feature matrix shapes:
  X_train: (800, 26)
  X_test:  (300, 26)

Training Random Forest classifier...

OVERALL PERFORMANCE
Training Accuracy: 1.000
Test Accuracy:     0.913
Test AUROC:        0.968

✓ Model trained successfully
Now let's examine performance BY SKIN TYPE...


## 4. Fairness Analysis: Performance by Skin Type

This is where the bias becomes apparent.

In [8]:
# Add predictions to test dataframe
df_test['pred'] = y_test_pred
df_test['pred_proba'] = y_test_proba

# Calculate metrics by skin group
print("="*80)
print("PERFORMANCE BY SKIN TYPE")
print("="*80)

results_by_group = []

for group in ['Light (I-II)', 'Medium (III-IV)', 'Dark (V-VI)']:
    df_group = df_test[df_test['skin_group'] == group]

    y_true = df_group['label'].values
    y_pred = df_group['pred'].values
    y_proba = df_group['pred_proba'].values

    # Calculate metrics
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    auroc = roc_auc_score(y_true, y_proba)

    results_by_group.append({
        'group': group,
        'n': len(df_group),
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'auroc': auroc
    })

    print(f"\n{group}:")
    print(f"  Sample size: {len(df_group)}")
    print(f"  Accuracy:    {accuracy:.3f}")
    print(f"  Sensitivity: {sensitivity:.3f} (ability to detect melanoma)")
    print(f"  Specificity: {specificity:.3f}")
    print(f"  AUROC:       {auroc:.3f}")

results_df = pd.DataFrame(results_by_group)

print("\n" + "="*80)
print("⚠️  DISPARITY ANALYSIS")
print("="*80)

# Calculate disparities
light_sens = results_df[results_df['group'] == 'Light (I-II)']['sensitivity'].values[0]
dark_sens = results_df[results_df['group'] == 'Dark (V-VI)']['sensitivity'].values[0]
sens_gap = light_sens - dark_sens

print(f"Sensitivity gap (Light - Dark): {sens_gap:.3f}")
n_dark_mel = int(((df_test['skin_group'] == 'Dark (V-VI)') & (df_test['label'] == 1)).sum())
if sens_gap > 0.05:
    print(f"\n💔 This means: for patients with dark skin, the model detected "
          f"{sens_gap * 100:.0f} percentage points fewer melanomas than for light skin.")
elif sens_gap > 0:
    print(f"\n🔎 Only a small gap (about {sens_gap * n_dark_mel:.0f} more missed melanoma(s) out of "
          f"{n_dark_mel} in the dark-skin test group): not a meaningful disparity at this sample size.")
else:
    print("\n🔎 No sensitivity gap against dark skin in this run.")
print("   Two causes are mixed here: lower lesion contrast on darker skin (built into the image")
print("   physics of the generator) and the 92/8 training imbalance. Section 5b separates them.")
if sens_gap <= 0.05:
    print("   On these simple colour features the classes are so easy to separate that imbalance has little effect.")
    print("   Exercise: make lesions less distinct (e.g. closer malignant/benign colours) and re-run.")
print("="*80)
print("Book (Priya's app): sensitivity 94% (I-II), 82% (III-IV), 67% (V-VI). Compare with the numbers above.")

PERFORMANCE BY SKIN TYPE

Light (I-II):
  Sample size: 100
  Accuracy:    0.990
  Sensitivity: 0.980 (ability to detect melanoma)
  Specificity: 1.000
  AUROC:       0.999

Medium (III-IV):
  Sample size: 100
  Accuracy:    0.940
  Sensitivity: 0.880 (ability to detect melanoma)
  Specificity: 1.000
  AUROC:       0.985

Dark (V-VI):
  Sample size: 100
  Accuracy:    0.810
  Sensitivity: 0.680 (ability to detect melanoma)
  Specificity: 0.940
  AUROC:       0.890

⚠️  DISPARITY ANALYSIS
Sensitivity gap (Light - Dark): 0.300

💔 This means: for patients with dark skin, the model detected 30 percentage points fewer melanomas than for light skin.
   Two causes are mixed here: lower lesion contrast on darker skin (built into the image
   physics of the generator) and the 92/8 training imbalance. Section 5b separates them.
Book (Priya's app): sensitivity 94% (I-II), 82% (III-IV), 67% (V-VI). Compare with the numbers above.


In [9]:
# Visualize disparities
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

groups = results_df['group'].values
x = np.arange(len(groups))
colors = ['#FFE4C4', '#DEB887', '#8B4513']

# Accuracy
ax = axes[0, 0]
ax.bar(x, results_df['accuracy'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=15, ha='right')
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy by Skin Type', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0.8, color='green', linestyle='--', alpha=0.5, label='Target')

# Sensitivity (most important - don't miss cancer!)
ax = axes[0, 1]
bars = ax.bar(x, results_df['sensitivity'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=15, ha='right')
ax.set_ylabel('Sensitivity', fontsize=12)
ax.set_title('Sensitivity by Skin Type\n(Ability to Detect Melanoma)', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0.9, color='green', linestyle='--', alpha=0.5, label='Target')

# Highlight the disparity
bars[2].set_edgecolor('red')
bars[2].set_linewidth(3)

# Specificity
ax = axes[1, 0]
ax.bar(x, results_df['specificity'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=15, ha='right')
ax.set_ylabel('Specificity', fontsize=12)
ax.set_title('Specificity by Skin Type', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# AUROC
ax = axes[1, 1]
ax.bar(x, results_df['auroc'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=15, ha='right')
ax.set_ylabel('AUROC', fontsize=12)
ax.set_title('AUROC by Skin Type', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

worst = results_df.loc[results_df['sensitivity'].idxmin()]
spread = results_df['sensitivity'].max() - results_df['sensitivity'].min()
if spread > 0.05:
    print(f"\n📊 Sensitivity is lowest for {worst['group']} ({worst['sensitivity']:.2f}); "
          f"spread across groups: {spread:.2f}")
else:
    print(f"\n📊 Sensitivity is similar across groups (spread {spread:.2f}): "
          "no clear degradation for dark skin in this synthetic run")


📊 Sensitivity is lowest for Dark (V-VI) (0.68); spread across groups: 0.30


### How stable are these numbers?

Only about 1 in 25 training images is type V-VI, i.e. a handful of dark-skin melanomas. Results for that group therefore swing from one training draw to the next. We redraw the (biased) training set 3 more times, keep the same test set, and report mean ± standard deviation.

In [10]:
groups_order = ['Light (I-II)', 'Medium (III-IV)', 'Dark (V-VI)']
def featurize(img):
    feats = []
    for c in range(3):
        hist, _ = np.histogram(img[:, :, c], bins=8, range=(0, 1))
        feats.extend(hist / hist.sum())
    feats.append(np.std(img))
    feats.append(np.std(img[:, :, 0] - img[:, :, 1]))
    return np.array(feats)

redraw_sens = []
for draw in range(3):
    df_redraw = generate_biased_dataset(n_samples=800, melanoma_rate=0.20)
    clf_redraw = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
    clf_redraw.fit(np.stack(df_redraw['features'].values), df_redraw['label'].values)
    pred_redraw = clf_redraw.predict(X_test)
    redraw_sens.append([pred_redraw[(df_test['skin_group'] == g).values & (y_test == 1)].mean() for g in groups_order])
all_sens = np.vstack([results_df.set_index('group').loc[groups_order, 'sensitivity'].values, redraw_sens])
book_sens = [0.94, 0.82, 0.67]
print(f"Sensitivity over {len(all_sens)} training draws (same test set):")
for j, g in enumerate(groups_order):
    print(f"  {g:16s}: {all_sens[:, j].mean():.2f} ± {all_sens[:, j].std():.2f}   (book: {book_sens[j]:.2f})")
dark_lowest = all_sens[:, 2].mean() < all_sens[:, 0].mean() - 0.1
print("→ Dark-skin sensitivity is consistently lower than light-skin sensitivity." if dark_lowest
      else "→ The light-dark difference is not consistent across draws.")
if all_sens[:, 2].std() > 1.5 * all_sens[:, 0].std():
    print("  The larger spread for V-VI is itself a lesson: a few training examples give an unstable model.")
else:
    print(f"  Spread is similar across groups here (sd {all_sens[:, 0].std():.2f} vs {all_sens[:, 2].std():.2f}); with more draws the")
    print("  V-VI estimate is still the least certain, since it rests on only a handful of training melanomas.")

Sensitivity over 4 training draws (same test set):
  Light (I-II)    : 0.91 ± 0.05   (book: 0.94)
  Medium (III-IV) : 0.91 ± 0.03   (book: 0.82)
  Dark (V-VI)     : 0.73 ± 0.05   (book: 0.67)
→ Dark-skin sensitivity is consistently lower than light-skin sensitivity.
  Spread is similar across groups here (sd 0.05 vs 0.05); with more draws the
  V-VI estimate is still the least certain, since it rests on only a handful of training melanomas.


## 5. Fairness Metrics

Quantifying fairness using standard metrics from the fairness ML literature.

In [11]:
def calculate_fairness_metrics(df, groups):
    """
    Calculate fairness metrics across demographic groups.
    """

    metrics = {}

    # 1. Demographic Parity (equal positive prediction rates)
    ppr_by_group = {}
    for group in groups:
        df_group = df[df['skin_group'] == group]
        ppr = df_group['pred'].mean()
        ppr_by_group[group] = ppr

    demographic_parity_diff = max(ppr_by_group.values()) - min(ppr_by_group.values())
    metrics['demographic_parity_diff'] = demographic_parity_diff

    # 2. Equalized Odds (equal TPR and FPR across groups)
    tpr_by_group = {}
    fpr_by_group = {}

    for group in groups:
        df_group = df[df['skin_group'] == group]
        cm = confusion_matrix(df_group['label'], df_group['pred'])
        tn, fp, fn, tp = cm.ravel()

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        tpr_by_group[group] = tpr
        fpr_by_group[group] = fpr

    tpr_diff = max(tpr_by_group.values()) - min(tpr_by_group.values())
    fpr_diff = max(fpr_by_group.values()) - min(fpr_by_group.values())

    metrics['equalized_odds_tpr_diff'] = tpr_diff
    metrics['equalized_odds_fpr_diff'] = fpr_diff
    metrics['equalized_odds_avg'] = (tpr_diff + fpr_diff) / 2

    # 3. Equal Opportunity (equal TPR - most important for healthcare)
    metrics['equal_opportunity_diff'] = tpr_diff

    return metrics, ppr_by_group, tpr_by_group, fpr_by_group

# Calculate fairness metrics
groups = ['Light (I-II)', 'Medium (III-IV)', 'Dark (V-VI)']
fairness_metrics, ppr, tpr, fpr = calculate_fairness_metrics(df_test, groups)

print("="*80)
print("FAIRNESS METRICS")
print("="*80)

print(f"\n1. Demographic Parity (positive prediction rate):")
for group, rate in ppr.items():
    print(f"   {group:20s}: {rate:.3f}")
print(f"   Difference (max-min):  {fairness_metrics['demographic_parity_diff']:.3f}")
print(f"   → Ideal: 0 (equal prediction rates)")

print(f"\n2. Equalized Odds (TPR and FPR equality):")
print(f"   True Positive Rate (Sensitivity):")
for group, rate in tpr.items():
    print(f"     {group:20s}: {rate:.3f}")
print(f"   TPR Difference:        {fairness_metrics['equalized_odds_tpr_diff']:.3f}")
print(f"\n   False Positive Rate:")
for group, rate in fpr.items():
    print(f"     {group:20s}: {rate:.3f}")
print(f"   FPR Difference:        {fairness_metrics['equalized_odds_fpr_diff']:.3f}")
print(f"   → Ideal: 0 for both (equal error rates)")

print(f"\n3. Equal Opportunity (TPR equality - most important for cancer):")
print(f"   TPR Difference:        {fairness_metrics['equal_opportunity_diff']:.3f}")
print(f"   → Ideal: 0 (equal chance to detect cancer)")

print("\n" + "="*80)
print("⚠️  INTERPRETATION")
print("="*80)
if fairness_metrics['equal_opportunity_diff'] > 0.1:
    print(f"❌ SIGNIFICANT BIAS DETECTED")
    print(f"   TPR difference of {fairness_metrics['equal_opportunity_diff']:.3f} means:")
    print(f"   Dark-skinned patients have substantially lower chance of melanoma detection")
    print(f"   This system SHOULD NOT be deployed without addressing bias!")
else:
    print(f"✓ Fair performance across groups")
print("="*80)

FAIRNESS METRICS

1. Demographic Parity (positive prediction rate):
   Light (I-II)        : 0.490
   Medium (III-IV)     : 0.440
   Dark (V-VI)         : 0.370
   Difference (max-min):  0.120
   → Ideal: 0 (equal prediction rates)

2. Equalized Odds (TPR and FPR equality):
   True Positive Rate (Sensitivity):
     Light (I-II)        : 0.980
     Medium (III-IV)     : 0.880
     Dark (V-VI)         : 0.680
   TPR Difference:        0.300

   False Positive Rate:
     Light (I-II)        : 0.000
     Medium (III-IV)     : 0.000
     Dark (V-VI)         : 0.060
   FPR Difference:        0.060
   → Ideal: 0 for both (equal error rates)

3. Equal Opportunity (TPR equality - most important for cancer):
   TPR Difference:        0.300
   → Ideal: 0 (equal chance to detect cancer)

⚠️  INTERPRETATION
❌ SIGNIFICANT BIAS DETECTED
   TPR difference of 0.300 means:
   Dark-skinned patients have substantially lower chance of melanoma detection
   This system SHOULD NOT be deployed without address

## 5b. Control: Does Balanced Training Close the Gap?

To separate the two causes, we train the same RandomForest on a training set of the same size (800) but with **equal numbers of each Fitzpatrick type**, and evaluate on the same balanced test set. If the gap narrows, the imbalance was part of the problem; whatever gap remains comes from the images themselves (lower contrast on darker skin), which more data of the same kind cannot fully fix.

In [12]:
def features_from_image(img):
    feats = []
    for c in range(3):
        hist, _ = np.histogram(img[:, :, c], bins=8, range=(0, 1))
        feats.extend(hist / hist.sum())
    feats.append(np.std(img))
    feats.append(np.std(img[:, :, 0] - img[:, :, 1]))
    return np.array(feats)

balanced_types = np.random.choice(np.arange(1, 7), size=800)            # uniform over types I-VI
balanced_labels = (np.random.rand(800) < 0.20).astype(int)
X_bal = np.stack([features_from_image(generate_synthetic_lesion(64, t, bool(l)))
                  for t, l in zip(balanced_types, balanced_labels)])
clf_bal = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
clf_bal.fit(X_bal, balanced_labels)
pred_bal = clf_bal.predict(X_test)
proba_bal = clf_bal.predict_proba(X_test)[:, 1]

rows = []
for group in groups:
    g = (df_test['skin_group'] == group).values
    rows.append({'group': group,
                 'sens_biased': df_test.loc[g & (y_test == 1), 'pred'].mean(),
                 'sens_balanced': pred_bal[g & (y_test == 1)].mean(),
                 'auroc_biased': roc_auc_score(y_test[g], y_test_proba[g]),
                 'auroc_balanced': roc_auc_score(y_test[g], proba_bal[g])})
control = pd.DataFrame(rows).set_index('group')
print(f"Training mix: biased {(df_train['skin_type'] >= 5).mean():.0%} types V-VI vs balanced {(balanced_types >= 5).mean():.0%}")
print(control.round(3).to_string())
print(f"Overall AUROC: biased {roc_auc_score(y_test, y_test_proba):.3f}, balanced {roc_auc_score(y_test, proba_bal):.3f}")

gap_biased = control.loc['Light (I-II)', 'sens_biased'] - control.loc['Dark (V-VI)', 'sens_biased']
gap_bal = control.loc['Light (I-II)', 'sens_balanced'] - control.loc['Dark (V-VI)', 'sens_balanced']
print(f"\nSensitivity gap (Light - Dark): biased training {gap_biased:.2f} → balanced training {gap_bal:.2f}")
if gap_bal < gap_biased - 0.05:
    print("→ Balanced training NARROWED the gap: the 92/8 imbalance was part of the problem.")
else:
    print("→ Balanced training did not clearly narrow the gap in this run.")
if gap_bal > 0.05:
    print("→ A gap REMAINS: on darker skin the lesion signal itself is weaker in these images, so")
    print("  fixing representation is necessary but not sufficient (better imaging/features are needed too).")
else:
    print("→ With balanced training the remaining gap is small.")

Training mix: biased 4% types V-VI vs balanced 34%
                 sens_biased  sens_balanced  auroc_biased  auroc_balanced
group                                                                    
Light (I-II)            0.98           0.94         0.999           1.000
Medium (III-IV)         0.88           0.94         0.985           0.979
Dark (V-VI)             0.68           0.90         0.890           0.984
Overall AUROC: biased 0.968, balanced 0.988

Sensitivity gap (Light - Dark): biased training 0.30 → balanced training 0.04
→ Balanced training NARROWED the gap: the 92/8 imbalance was part of the problem.
→ With balanced training the remaining gap is small.


## 6. Priya's Case: What Went Wrong

Let's simulate what happened to Priya.

In [13]:
# Simulate Priya's melanoma
priya_img = generate_synthetic_lesion(size=64, skin_type=4, is_malignant=True)  # Fitzpatrick type IV

# Extract features
priya_features = []
for c in range(3):
    hist, _ = np.histogram(priya_img[:, :, c], bins=8, range=(0, 1))
    priya_features.extend(hist / hist.sum())
priya_features.append(np.std(priya_img))
priya_features.append(np.std(priya_img[:, :, 0] - priya_img[:, :, 1]))

# Model prediction
priya_pred_proba = clf.predict_proba([priya_features])[0, 1]
priya_pred = clf.predict([priya_features])[0]

# For comparison: light-skinned patient with same melanoma
comparison_img = generate_synthetic_lesion(size=64, skin_type=1, is_malignant=True)
comparison_features = []
for c in range(3):
    hist, _ = np.histogram(comparison_img[:, :, c], bins=8, range=(0, 1))
    comparison_features.extend(hist / hist.sum())
comparison_features.append(np.std(comparison_img))
comparison_features.append(np.std(comparison_img[:, :, 0] - comparison_img[:, :, 1]))

comparison_pred_proba = clf.predict_proba([comparison_features])[0, 1]
comparison_pred = clf.predict([comparison_features])[0]

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Priya's case
axes[0].imshow(priya_img)
axes[0].set_title(f'Priya (Fitzpatrick Type IV)\nMelanoma\n\nAI Prediction: {priya_pred_proba:.2%} risk\n{"HIGH RISK" if priya_pred == 1 else "LOW RISK (MISSED)"}',
                 fontweight='bold', fontsize=12,
                 color='green' if priya_pred == 1 else 'red')
axes[0].axis('off')

# Comparison case
axes[1].imshow(comparison_img)
axes[1].set_title(f'Comparison (Fitzpatrick Type I)\nMelanoma\n\nAI Prediction: {comparison_pred_proba:.2%} risk\n{"HIGH RISK (DETECTED)" if comparison_pred == 1 else "LOW RISK"}',
                 fontweight='bold', fontsize=12,
                 color='green' if comparison_pred == 1 else 'red')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("="*80)
print("JOURNEY 5: PRIYA'S CASE")
print("="*80)
print(f"\nPriya (Type IV, medium-dark skin; 'Medium (III-IV)' group here):")
print(f"  True diagnosis: MELANOMA")
print(f"  AI prediction:  {priya_pred_proba:.1%} risk")
print(f"  Result:         {'DETECTED' if priya_pred == 1 else 'MISSED - Low risk'}")

print(f"\nComparison patient (Type I light skin):")
print(f"  True diagnosis: MELANOMA (same)")
print(f"  AI prediction:  {comparison_pred_proba:.1%} risk")
print(f"  Result:         {'DETECTED' if comparison_pred == 1 else 'MISSED'}")

print(f"\n💔 What happened to Priya in the case story:")
print(f"   1. The app called her shoulder lesion 'Low risk, benign nevus' (confidence 87%)")
print(f"   2. She skipped the dermatologist (trusted the app)")
print(f"   3. Three months later: melanoma, stage IB (1.2 mm depth)")
print(f"   4. Wider surgical margins, worse prognosis than if caught earlier")

if priya_pred == 1:
    print(f"\n🔎 Our synthetic model DETECTED simulated Priya ({priya_pred_proba:.0%} risk), unlike the app in the story.")
    print(f"   One simulated case can go either way; the group sensitivities above are what matter.")
else:
    print(f"\n⚠️  Our synthetic model also MISSED simulated Priya ({priya_pred_proba:.0%} risk).")
print(f"\n⚠️  ROOT CAUSE (in the story):")
print(f"   Training data skewed to light skin (book: 92% types I-III; here: {(df_train['skin_type'] <= 3).mean():.0%} I-III, {(df_train['skin_type'] >= 4).mean():.0%} IV-VI)")
print(f"   A model can learn features that work well on light skin but fail on dark skin")
print(f"   → Algorithmic bias can cause real harm")
print("="*80)

JOURNEY 5: PRIYA'S CASE

Priya (Type IV, medium-dark skin; 'Medium (III-IV)' group here):
  True diagnosis: MELANOMA
  AI prediction:  99.0% risk
  Result:         DETECTED

Comparison patient (Type I light skin):
  True diagnosis: MELANOMA (same)
  AI prediction:  92.0% risk
  Result:         DETECTED

💔 What happened to Priya in the case story:
   1. The app called her shoulder lesion 'Low risk, benign nevus' (confidence 87%)
   2. She skipped the dermatologist (trusted the app)
   3. Three months later: melanoma, stage IB (1.2 mm depth)
   4. Wider surgical margins, worse prognosis than if caught earlier

🔎 Our synthetic model DETECTED simulated Priya (99% risk), unlike the app in the story.
   One simulated case can go either way; the group sensitivities above are what matter.

⚠️  ROOT CAUSE (in the story):
   Training data skewed to light skin (book: 92% types I-III; here: 93% I-III, 7% IV-VI)
   A model can learn features that work well on light skin but fail on dark skin
   → A

## 7. Transfer Learning: Pretrain → Freeze → Fine-tune

The book's recipe (Section 6.2) is **transfer learning**: start from a CNN pretrained on a large, generic image set, keep (freeze) its convolutional layers, which have learned generic edges, textures and colour patterns, and train only a new classification head on the small medical dataset. Optionally, then **fine-tune** the last convolutional block with a small learning rate.

In practice the pretrained network is an ImageNet model such as `tf.keras.applications.MobileNetV2(weights='imagenet')`. That needs a weights download, so this notebook uses a **stand-in that runs offline**:

1. **Pretrain** a small CNN on a *related synthetic source task* with many examples: 4,000 images, 4 classes = {small, large} × {uniform, variegated} blobs of random colours on **random-coloured backgrounds of all brightness levels** (no skin, no melanoma labels). Like ImageNet for dermoscopy, it shares low-level structure (blobs, borders, colour variation) with the target, but it is deliberately closer than ImageNet, i.e. a best case for transfer.
2. **Freeze** the convolutional layers and train a new head on the melanoma images; then unfreeze the last conv layer and fine-tune briefly.
3. **Learning curve**: train on 50, 100, 200 and 400 images drawn from the biased training set (same 92% / 8% skin-type mix), and compare with the same CNN **trained from scratch** and with the **RandomForest**.
4. **Operating point**: each model's decision threshold is chosen on a separate **validation split** (200 other images from the biased training distribution, maximising sensitivity + specificity), never on the test set. We then report **sensitivity and specificity per skin group** on the balanced test set.

All data are synthetic, so the numbers show how the pattern works, not how a real dermatology model performs.

*This section needs TensorFlow (Colab or local Jupyter). In the browser version it is skipped and the rest of the notebook still runs.*

In [14]:
try:
    import tensorflow as tf
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("ℹ️ This section needs TensorFlow (Colab or local Jupyter); it is skipped in the browser version.")

def to_cnn_input(images):
    """Downsample the 64×64 RGB lesions to 32×32 to keep training fast on a CPU."""
    return np.stack([img[::2, ::2] for img in images]).astype('float32')

if HAS_TF:
    tf.keras.utils.set_random_seed(42)
    rng_src = np.random.default_rng(7)

    def generate_source_image(size=32):
        """Source task: 4 classes = {small, large} x {uniform, variegated} blobs on random backgrounds."""
        background = rng_src.uniform(0.1, 0.95, 3) * rng_src.uniform(0.4, 1.0)   # all brightness levels
        img = background + rng_src.normal(0, 0.05, (size, size, 3))
        yy, xx = np.mgrid[:size, :size]
        large, variegated = rng_src.integers(0, 2), rng_src.integers(0, 2)
        r = rng_src.uniform(7, 10) if large else rng_src.uniform(4, 6)
        cy, cx = size / 2 + rng_src.uniform(-3, 3, 2)
        e = rng_src.uniform(0.8, 1.25)
        mask = ((yy - cy) / (r * e))**2 + ((xx - cx) / (r / e))**2 <= 1
        amp = rng_src.uniform(0.25, 0.5) if variegated else rng_src.uniform(0, 0.04)
        shade = np.clip(1 + 3 * amp * gaussian_filter(rng_src.standard_normal((size, size)), 0.75), 0.2, 1.5)
        colour = rng_src.uniform(0.05, 0.9, 3)
        img[mask] = (colour * shade[..., None])[mask] + rng_src.normal(0, 0.05, (mask.sum(), 3))
        return np.clip(img, 0, 1), 2 * large + variegated

    src = [generate_source_image() for _ in range(4000)]
    X_src = np.stack([s[0] for s in src]).astype('float32')
    y_src = np.array([s[1] for s in src])

    def build_conv_base():
        return tf.keras.Sequential([
            tf.keras.layers.Input((32, 32, 3)),
            tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same', name='last_conv'),
            tf.keras.layers.GlobalAveragePooling2D(),
        ], name='conv_base')

    # Step 1: pretrain on the source task
    source_base = build_conv_base()
    source_model = tf.keras.Sequential([source_base, tf.keras.layers.Dense(4, activation='softmax')])
    source_model.compile(optimizer=tf.keras.optimizers.Adam(3e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = source_model.fit(X_src, y_src, epochs=18, batch_size=64, validation_split=0.15, verbose=0)
    print(f"Step 1 - source task (4 classes): validation accuracy {hist.history['val_accuracy'][-1]:.3f} "
          f"(chance 0.25)")
    pretrained_weights = source_base.get_weights()

    # Target data: train subsets from the biased set, a separate validation split, the balanced test set
    X_cnn_all = to_cnn_input(df_train['image'])
    X_cnn_test = to_cnn_input(df_test['image'])
    VAL = slice(600, 800)                     # validation split (never used for training)
    X_cnn_val, X_rf_val, y_val = X_cnn_all[VAL], X_train[VAL], y_train[VAL]

    def youden_threshold(y_true, proba):
        fpr_v, tpr_v, thr = roc_curve(y_true, proba)
        return thr[np.argmax(tpr_v - fpr_v)]

    def fit_models(n, fine_tune=False):
        """Train RF, scratch CNN and transfer CNN on the first n biased training images.
        Returns {model name: (validation probabilities, test probabilities)}."""
        Xn, yn = X_cnn_all[:n], y_train[:n]
        cw = {0: 1.0, 1: float((yn == 0).sum() / max(yn.sum(), 1))}   # up-weight the rarer melanomas
        out = {}
        rf_n = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
        rf_n.fit(X_train[:n], yn)
        out['RandomForest'] = (rf_n.predict_proba(X_rf_val)[:, 1], rf_n.predict_proba(X_test)[:, 1])

        scratch = tf.keras.Sequential([build_conv_base(), tf.keras.layers.Dense(1, activation='sigmoid')])
        scratch.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
        scratch.fit(Xn, yn, epochs=30, batch_size=32, class_weight=cw, verbose=0)
        out['CNN from scratch'] = (scratch.predict(X_cnn_val, verbose=0).ravel(), scratch.predict(X_cnn_test, verbose=0).ravel())

        base = build_conv_base()
        base.set_weights(pretrained_weights)
        base.trainable = False                                       # Step 2: freeze
        tl = tf.keras.Sequential([base, tf.keras.layers.Dense(1, activation='sigmoid')])
        tl.compile(optimizer=tf.keras.optimizers.Adam(1e-2), loss='binary_crossentropy')
        tl.fit(Xn, yn, epochs=30, batch_size=32, class_weight=cw, verbose=0)
        out['CNN transfer (frozen)'] = (tl.predict(X_cnn_val, verbose=0).ravel(), tl.predict(X_cnn_test, verbose=0).ravel())
        if fine_tune:                                                # Step 2b: unfreeze last conv layer
            base.trainable = True
            for layer in base.layers:
                layer.trainable = layer.name == 'last_conv'
            tl.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy')
            tl.fit(Xn, yn, epochs=15, batch_size=32, class_weight=cw, verbose=0)
            out['CNN transfer + fine-tune'] = (tl.predict(X_cnn_val, verbose=0).ravel(), tl.predict(X_cnn_test, verbose=0).ravel())
        return out

    # Step 3: learning curve
    sizes = [50, 100, 200, 400]
    curve, results_at = {}, {}
    for n in sizes:
        results_at[n] = fit_models(n, fine_tune=(n == 200))
        for name, (pv, pt) in results_at[n].items():
            curve.setdefault(name, {})[n] = roc_auc_score(y_test, pt)
        print(f"  trained on {n:3d} images: " +
              ", ".join(f"{k} AUROC {curve[k][n]:.3f}" for k in ['RandomForest', 'CNN from scratch', 'CNN transfer (frozen)']))

I0000 00:00:1795590096.617914    2703 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795590096.618555    2703 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795590097.994023    2703 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795590097.994389    2703 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Step 1 - source task (4 classes): validation accuracy 0.917 (chance 0.25)


  trained on  50 images: RandomForest AUROC 0.868, CNN from scratch AUROC 0.615, CNN transfer (frozen) AUROC 0.990


  trained on 100 images: RandomForest AUROC 0.908, CNN from scratch AUROC 0.548, CNN transfer (frozen) AUROC 0.990


  trained on 200 images: RandomForest AUROC 0.945, CNN from scratch AUROC 0.726, CNN transfer (frozen) AUROC 0.988


  trained on 400 images: RandomForest AUROC 0.969, CNN from scratch AUROC 0.932, CNN transfer (frozen) AUROC 0.988


In [15]:
if HAS_TF:
    fig, ax = plt.subplots(figsize=(9, 5))
    for name, style in [('RandomForest', 's--'), ('CNN from scratch', 'o-'), ('CNN transfer (frozen)', '^-')]:
        ax.plot(sizes, [curve[name][n] for n in sizes], style, linewidth=2, markersize=8, label=name)
    ax.set_xscale('log')
    ax.set_xticks(sizes)
    ax.set_xticklabels(sizes)
    ax.set_xlabel('Number of melanoma-task training images (biased skin-type mix)')
    ax.set_ylabel('Test AUROC (balanced test set)')
    ax.set_ylim(0.4, 1.02)
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
    ax.set_title('Learning curves: transfer learning vs training from scratch (synthetic data)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Per-group operating point at n = 200, thresholds chosen on the validation split
    N_MAIN = 200
    rows = []
    for name, (pv, pt) in results_at[N_MAIN].items():
        thr = youden_threshold(y_val, pv)
        pred = (pt >= thr).astype(int)
        row = {'model': name, 'threshold': thr, 'AUROC': roc_auc_score(y_test, pt)}
        for g in groups:
            in_g = (df_test['skin_group'] == g).values
            row[f'Sens {g.split()[0]}'] = pred[in_g & (y_test == 1)].mean()
            row[f'Spec {g.split()[0]}'] = 1 - pred[in_g & (y_test == 0)].mean()
        rows.append(row)
    comparison = pd.DataFrame(rows).set_index('model')
    print(f"Trained on {N_MAIN} images; threshold from validation split; balanced synthetic test set:")
    print(comparison.round(2).to_string())
    print("Book (Priya's app): sensitivity 0.94 (Light), 0.82 (Medium), 0.67 (Dark)")

    print("\n🔎 What the numbers say:")
    gain = {n: curve['CNN transfer (frozen)'][n] - curve['CNN from scratch'][n] for n in sizes}
    for n in sizes:
        print(f"   n={n:3d}: transfer − scratch AUROC = {gain[n]:+.3f}")
    if gain[50] > 0.1 and gain[50] > gain[400]:
        print("   → Transfer learning helps most when the target set is small; the advantage shrinks as data grow.")
    elif gain[50] > 0.1:
        print("   → Transfer learning clearly beats training from scratch on small target sets.")
    else:
        print("   → In this run transfer learning did not clearly beat training from scratch.")
    tl_auc, rf_auc = curve['CNN transfer (frozen)'][N_MAIN], curve['RandomForest'][N_MAIN]
    print(f"   At n={N_MAIN}: transfer {tl_auc:.3f} vs RandomForest {rf_auc:.3f} AUROC "
          f"({'transfer is at least as good' if tl_auc >= rf_auc - 0.01 else 'RandomForest is better'}).")
    if 'CNN transfer + fine-tune' in comparison.index:
        ft = comparison.loc['CNN transfer + fine-tune', 'AUROC'] - comparison.loc['CNN transfer (frozen)', 'AUROC']
        print(f"   Fine-tuning the last conv layer changed AUROC by {ft:+.3f}.")
    for name in comparison.index:
        sens = [comparison.loc[name, f'Sens {g.split()[0]}'] for g in groups]
        spec = [comparison.loc[name, f'Spec {g.split()[0]}'] for g in groups]
        if min(spec) < 0.5 or min(sens) < 0.5:   # e.g. flags most lesions in some group: not a usable operating point
            print(f"   ⚠️ {name}: degenerate in at least one group (sens {np.round(sens, 2)}, spec {np.round(spec, 2)}).")
    best = 'CNN transfer + fine-tune' if 'CNN transfer + fine-tune' in comparison.index else 'CNN transfer (frozen)'
    s_light, s_dark = comparison.loc[best, 'Sens Light'], comparison.loc[best, 'Sens Dark']
    print(f"   {best}: sensitivity Light {s_light:.2f} vs Dark {s_dark:.2f} "
          f"(specificity Dark {comparison.loc[best, 'Spec Dark']:.2f}).")
    if s_light - s_dark > 0.05:
        print("   ⚠️ Even the best model detects fewer melanomas on dark skin: a better feature extractor does not")
        print("      remove the effect of lower lesion contrast and a 4% dark-skin training share. Stratify, always.")
    else:
        print("   The dark-skin gap is small for this model in this run; on real data, always stratify anyway.")
else:
    print("Transfer-learning comparison skipped (TensorFlow not available).")

Trained on 200 images; threshold from validation split; balanced synthetic test set:
                          threshold  AUROC  Sens Light  Spec Light  Sens Medium  Spec Medium  Sens Dark  Spec Dark
model                                                                                                             
RandomForest                   0.34   0.95        0.94        1.00         0.88         1.00       0.82       0.68
CNN from scratch               0.60   0.73        0.58        0.86         0.58         0.78       0.88       0.26
CNN transfer (frozen)          0.34   0.99        1.00        0.94         0.92         0.96       0.90       0.98
CNN transfer + fine-tune       0.34   0.99        1.00        0.94         0.92         0.96       0.90       0.98
Book (Priya's app): sensitivity 0.94 (Light), 0.82 (Medium), 0.67 (Dark)

🔎 What the numbers say:
   n= 50: transfer − scratch AUROC = +0.375
   n=100: transfer − scratch AUROC = +0.442
   n=200: transfer − scratch AUROC = +0

## 8. Key Takeaways

### What We Learned

1. **Algorithmic bias is real and harmful**:
   - Training data bias leads to performance disparities
   - In Priya's app (book): sensitivity 94% (I-II) vs 82% (III-IV) vs 67% (V-VI)
   - This is not a "bug" - it's a systemic problem

2. **Fairness metrics are essential**:
   - **Demographic parity**: Equal positive prediction rates
   - **Equalized odds**: Equal TPR and FPR across groups
   - **Equal opportunity**: Equal TPR (most important for cancer screening)
   - Must evaluate stratified by demographic groups

3. **Data representativeness matters**:
   - Priya's app: 92% of training images from types I-III, 8% from IV-VI
   - This mirrors historical inequities in medical research
   - Diverse training data is necessary (but not sufficient)

4. **Clinical harm from bias**:
   - Missed diagnoses lead to worse outcomes
   - Lower sensitivity → delayed treatment → advanced disease
   - Disproportionate harm to already underserved populations

5. **Solutions require multiple approaches**:
   - Diverse training data collection
   - Stratified evaluation during development
   - Post-deployment monitoring by demographic group
   - Transparency about performance limitations
   - Clinical oversight, not autonomous decisions

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: Priya's story provides the clinical motivation
- **Chapter 4 (Data Preparation)**: Data collection, representativeness, sampling bias
- **Chapter 5 (Evaluation)**: Stratified metrics, performance disparities
- **Chapter 6 (Imaging)**: This chapter - transfer learning (pretrain → freeze → fine-tune), feature extraction
- **Chapter 9 (Fairness & Bias)**: Deep dive into fairness metrics, mitigation strategies
- **Chapter 11 (Deployment)**: Monitoring for bias, regulatory considerations

### Real-World Context

**Research findings**:
- Multiple studies show lower accuracy on dark skin (Adamson & Smith 2018, Daneshjou et al. 2022)
- ISIC dataset: <10% images of skin types IV-VI
- FDA has issued guidance on bias in AI/ML medical devices

**Mitigation strategies**:
1. **Data**: Collect diverse, representative datasets (Fitzpatrick17k, DDI)
2. **Algorithms**: Fairness-aware training, re-weighting, adversarial debiasing
3. **Evaluation**: Mandatory stratified reporting by skin tone
4. **Deployment**: Clinical oversight, clear performance limitations
5. **Regulation**: FDA requires bias assessment for AI devices

**Why this is hard**:
- Historical underrepresentation in medical research
- Data collection challenges (privacy, consent, access)
- No single "fix" - requires systemic change
- Trade-offs between fairness metrics

---

## Exercises

1. **Rebalanced training**: Create a training set with equal representation of all skin types. Does this improve fairness? What happens to overall performance?

2. **Fairness constraints**: Implement a training procedure that optimizes for both accuracy AND equal opportunity. What's the trade-off?

3. **Threshold optimization**: Can you select different thresholds for different skin types to achieve equal sensitivity? Is this ethical?

4. **Intersectionality**: Add age and gender as factors. Are there intersectional disparities (e.g., dark-skinned women vs dark-skinned men)?

5. **Economic analysis**: Calculate the cost of missed melanomas (treatment, outcomes) across groups. What's the total societal cost of biased AI?

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 6: Medical Imaging)*  
*This implements Journey 5 (Priya - Melanoma Fairness) from Chapter 3*  
*For clinical context, see Chapter 3. For fairness methods, see Chapter 9.*